## Embadding Vector stores
* Chroma
* FAISS
* 

In [3]:
from langchain_classic.document_loaders import PyPDFLoader, TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

/home/mysudarshan/Documents/AIML/NLP/envNLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

<B>Step 1.</B>  Lets Load First

In [5]:
pdfLoader = PyPDFLoader(file_path="Docs/ShyamSundarResume.pdf")
resume_doc = pdfLoader.load()
resume_doc

[Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-08-06T09:52:10+05:30', 'source': 'Docs/ShyamSundarResume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='SHYAM SUNDAR\nSoftware Engineer @ Accenture \nEmail: ssundar.kars@gmail.com || Ph.: +91 9555218287\nGithub:ssundarkars || LinkedIn:shyam-sundar-326b5b1b0/  || Location: Jalaun (U.P.)\nPROFESSIONAL SUMMARY\nSoftware Engineer with 2 years of experience developing scalable enterprise applications and integrating AI-driven \nautomation solutions in Agile environments. Experienced in full-cycle enterprise software development, with a strong, hands-on focus \non Python, Generative AI, Machine Learning, and building Retrieval-Augmented Generation (RAG) pipelines. Proven ability to \ndeliver business-critical software while leveraging modern AI frameworks and deep learning fundamentals to automate complex \nworkflows and solve technical challenges.\nLanguages: Python, SAP ABAP

<B>Step 2.</B> Get the data in to chunks (Easy for processing)

In [6]:
splitter = RecursiveCharacterTextSplitter(
    separators= ['\n\n', '\n'],
    chunk_size = 500,
    chunk_overlap = 40 
)

resume_chunks = splitter.split_documents(resume_doc)
len(resume_chunks), resume_chunks


(14,
 [Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-08-06T09:52:10+05:30', 'source': 'Docs/ShyamSundarResume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='SHYAM SUNDAR\nSoftware Engineer @ Accenture \nEmail: ssundar.kars@gmail.com || Ph.: +91 9555218287\nGithub:ssundarkars || LinkedIn:shyam-sundar-326b5b1b0/  || Location: Jalaun (U.P.)\nPROFESSIONAL SUMMARY\nSoftware Engineer with 2 years of experience developing scalable enterprise applications and integrating AI-driven \nautomation solutions in Agile environments. Experienced in full-cycle enterprise software development, with a strong, hands-on focus'),
  Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-08-06T09:52:10+05:30', 'source': 'Docs/ShyamSundarResume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='on Python, Generative AI, Machine Learning, and building Retrieval-Augmented Generation (RAG

<B>Step 3.</B> Embadding genreration
* 3.1 : Embadding Model

In [7]:
# Model preprration for embaddings
embed_model = HuggingFaceEmbeddings(model= "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 320.52it/s]


* 3.2: Embedding in Vector Store

In [8]:
from langchain_chroma import Chroma

chroma_store = Chroma.from_documents(resume_chunks, embed_model)

chroma_store


In [21]:
query = "Education"

In [22]:
query_embed = embed_model.embed_query(query)
result = chroma_store.similarity_search_with_score(query,3)
result

[(Document(id='b93b42dd-fdf1-45c7-bb30-411bb122fb38', metadata={'page': 1, 'creator': 'Writer', 'producer': 'LibreOffice 24.2', 'total_pages': 2, 'creationdate': '2026-08-06T09:52:10+05:30', 'source': 'Docs/ShyamSundarResume.pdf', 'page_label': '2'}, page_content='Bachelor  of  Technology  (B.Tech)  in  Computer  Science  and \nEngineering: 8.08 CGPA\nVidyagyan (U.P .)      | Jul 2013 – Apr 2020 (on Scholarship) \nIntermediate (2019 – 2020): 91.80% \nHigh School (2017 – 2018): 93.40%\nACHIEVEMENTS & LEADERSHIP\nZonal Badminton Tournament (2018,2019)\nVice Sports Captain (2018-2019)\nCodechef : 500+ Questions solved, 80+ Contest\nLeetCode:  250+ Questions Solved\nAvailability : Immediate'),
  1.4446983337402344),
 (Document(id='9e22d39b-c2ab-4e62-b0a9-d8d6158beaf3', metadata={'page': 0, 'total_pages': 2, 'source': 'Docs/ShyamSundarResume.pdf', 'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'page_label': '1', 'creationdate': '2026-08-06T09:52:10+05:30'}, page_content='AI/ML: Genera

In [23]:
for docs in result:
    print(docs[0].page_content, sep='\n\n')
    print()

Bachelor  of  Technology  (B.Tech)  in  Computer  Science  and 
Engineering: 8.08 CGPA
Vidyagyan (U.P .)      | Jul 2013 – Apr 2020 (on Scholarship) 
Intermediate (2019 – 2020): 91.80% 
High School (2017 – 2018): 93.40%
ACHIEVEMENTS & LEADERSHIP
Zonal Badminton Tournament (2018,2019)
Vice Sports Captain (2018-2019)
Codechef : 500+ Questions solved, 80+ Contest
LeetCode:  250+ Questions Solved
Availability : Immediate

AI/ML: Generative AI, Agentic AI, RAG, Computer Vision, Image Processing, NLP, LLP
Cloud & Systems: Docker, Kubernetes, Load Balancers, GCP, Linux, System Design (HLD & LLD), Git
Databases: Vector(FAISS, ChromaDB, Pinecone), MySQL, MongoDB, SAP S/4HANA
WORK EXPERIENCE
Analyst – Accenture                                                                                                                                            Aug 2024 – July 2026

 Built an automated attendance and authentication system using facial recognition and machine learning models (OpenCV).
 Engin